In [21]:
# this will translate parseTrials.m and conditionParsing.m from DRK npix analysis code

# spike_rate_gauss will take the binary spikes and convert to a smooth firing rate
# parse_trials will build the binary timelines, sort by trial, and shuffle to get significance thresholds
# parse_trials converts the nwb data from seconds to ms, then in spike_rate_gauss it's converted back to sec

#parse_conditions will... TO DO 

# we need numpy and the the 1-dimensional gaussian filter from scipy
# ndimage is just scipy's multi-D image processing
import numpy as np
from scipy.ndimage import gaussian_filter1d





In [ ]:
# We need to translate spikeRateGauss from Matlab. 
# It applies a Gaussian kernel to convert binary spike train into smooth firing rate
# We need to make a function that takes in the spike train and applies gaussian_filter1d

def spike_rate_gauss(spike_train, sigma=25):
    """
    Smooth a spike train (0s and 1s) into Hz firing rate
    spike_train: numpy array of 0s and 1s, one value per millisecond
    sigma: width of the Gaussian kernel in ms (default 25ms)

    returns: numpy array of smoothed firing rate in Hz (spikes/s).
    """
    smoothed = gaussian_filter1d(spike_train.astype(float), sigma=sigma) 
    return smoothed * 1000 # converts from spikes/ms to spikes/s





In [ ]:
# This will translate parseTrials.m
# For each recording site, we need to build a binary spike train (to go into spike_rate_gauss)
# 0 = no spike, 1 = spike, at 1ms resolution
# for each trial, cut a window around fixation (ITI), image onset, and response (button press)
# then shuffle to get significance thresholds
# Average across trials to get mean firing rate per recording site

def parse_trials(unit_spikes, trials_df, interval=None,
                 end_trials=None, first_spike_time=None, last_spike_time=None,
                 shuffle_fr=True, x_shuffle=100, alph=0.05, sig_window=30,
                 shuffle_hist=None):
    """
    Cuts spike data into trial-aligned windows around ITI, stimulus, and response.

    Inputs:
    unit_spikes: list of numpy arrays, each array is spike times (ms) for one recording site.
    trials_df: pandas df with trial timing info (defined in BASICPROC notebook)
    interval: time window sizes in ms around each event 
    end_trials: defines which trial to stop at
    first_spike_time, last_spike_time: start and end of recording (seconds)
    shuffle_fr: True/False whether to run shuffle test 
    x_shuffle: how many shuffles to run 

    Outputs:
    spk_iti, spk_stim_on, spk_resp: numpy arrays that contain the trial-aligned spike data
    shuffle_hist: if shuffle_fr = true this is the significance thresholds for each recording site
    """

    # if shuffle_hist is provided, we don't run the shuffle test.
    if shuffle_hist is not None:
        shuffle_fr = False
    
    # compute significance thresholds if shuffle_fr is True
    # e.g. 100 shuffles and alpha of 0.05
        # the alphaPThreshold = index number 98 = top 2.5% = positive threshold
        # the alphaNThreshold = index number 2 = bottom 2.5% = negative threshold
        # two-tailed significance therefore we split alpha in half for each tail
        # must round to integer index since Python array is in integers
    alpha_p_threshold = round(x_shuffle * (1-alph/2))
    alpha_n_threshold = round(x_shuffle * (alph/2))

    # if no endTrials is provided, use all trials
    if end_trials is None:
        end_trials = len(trials_df)

    # if first/last spike time is not provided, use min/max across all units
    # the for loops here find min and max spike times across all units
    # e.g., the min(spikes[0]) function finds the first spike time in each unit, iterated across all units, and takes the lowest one
    if last_spike_time is None:
        first_spike_time = min(spikes[0] for spikes in unit_spikes) #first spike
        last_spike_time = max(spikes[-1] for spikes in unit_spikes) #last spike, in python -1 is the last element of the array

    # Set default interval if not provided
    # using a Python dict: instead of defining multiple variables individually,
    # we can store them in a dict so that later on when we reuse something like 'prestim',
    # everything is in one place under the 'interval' variable.
    # these will be trimmed 250ms from each edge so for example the stim window will actually be 500ms before, 1500ms after, etc.
    if interval is None:
        interval = {
            'pre_stim': 750, #capture 750ms before stim onset (for edge effects, same below)
            'post_stim': 1750, #capture 1750ms after stim onset
            'pre_iti': 250, 'post_iti': 2500, #shortest ITI is 2s, although we will likely take middle of ITI (?)
            'pre_resp': 1250, 'post_resp': 750
        }
    pre_stim = interval['pre_stim']
    post_stim = interval['post_stim']
    pre_iti = interval['pre_iti']
    post_iti = interval['post_iti']
    pre_resp = interval['pre_resp']
    post_resp = interval['post_resp']



    # We need output dicts that will later be filled with the cut windows
    # spk_time: the x-axis, will just have time info around stim: e.g. [-749, -748, ..., 0, ..., 1749]
    # 0 = moment of the event (stimulus on, fixation on, or button)
    # np.arrange generates sequence of integers, from start to end (not including end so we add 1)
    # spk: binary data from unit_spikes (cut by window), will be a list of 2d matrices, 1 matrix per unit
    # spk_rate_sm: same as spk but smoothed firing rate (after applying spike_rate_gauss)
    # FOR FUTURE CALLING: these are indexed as [unit][trial, ms]
    # once we average across trials, we'll have a mean firing rate for each unit, which becomes the PSTH!
    spk_stim_on = {'spk_time': np.arange(-pre_stim + 1, post_stim + 1), 'spk': [], 'spk_rate_sm': []}
    spk_iti = {'spk_time': np.arange(-pre_iti  + 1, post_iti  + 1), 'spk': [], 'spk_rate_sm': []}
    spk_resp = {'spk_time': np.arange(-pre_resp + 1, post_resp + 1), 'spk': [], 'spk_rate_sm': []}
    
    #Create template/blank grids (trials x ms)
    #These will be copied in the loop below, to match the number of units (1 grid per unit)
    #Rows = trials, columns = ms in window
    n_trials = end_trials
    temp_stim_size = np.zeros((n_trials, pre_stim + post_stim))
    temp_iti_size = np.zeros((n_trials, pre_iti + post_iti))
    temp_resp_size = np.zeros((n_trials, pre_resp + post_resp))

    # these are separate arrays to store info that will help determine the windows to cut
    iti_end = np.zeros(n_trials) # 1D array to later store end time of each ITI
    response_time = np.zeros(n_trials) # 1D array to later store when pt actually pressed button
    # shuffle_hist will store the shuffle results if shuffle_fr is True. 
    # this just fills the array with None for now. 
    shuffle_hist = shuffle_hist if shuffle_hist is not None else [None] * len(unit_spikes)

    # Now we loop through each recording site/unit
    # enumerate() gives us both index "jj" and spike array
    # "spikes" here is just a temporary label used in the loop
    # Python arrays are 0-indexed so we subtract 1
    for jj, spikes in enumerate(unit_spikes):
        n_samples = round(last_spike_time * 1000) # total length of recording in ms
        spike_train = np.zeros(n_samples) # blank array, one slot per ms

        # convert sec to ms, adjust for 0-indexing
        # this is just a list of index positions that tells us where in spike_train to place 1s. 
        # e.g., if a spike fires at 1.358 sec, it's converted to 1358 ms,
        # but we subtract 1 to make the index pos. 1357, b/c Python starts at zero,
        # so the 1357 position is the 1358th ms, etc...
        spike_times_ms = np.round(spikes * 1000).astype(int) - 1 
        spike_times_ms = spike_times_ms[(spike_times_ms >= 0) & (spike_times_ms < n_samples)]  # clip to valid range
        spike_train[spike_times_ms] = 1 # place 1 at each spike position for each valid spike

        # now we will smooth the binary spike train using spike_rate_gauss
        smooth_rate = spike_rate_gauss(spike_train) #result: smoothed firing rate in Hz

        # now in the loop, we .copy() temp_stim_size etc from earlier, to make one template for each unit
        # each of these once filled later will be appended onto the output dicts (like spk_stim_on above)
        spk_mat_stim = temp_stim_size.copy()  # will hold raw spikes 0 or 1: [trials x ms]
        spk_mat_iti  = temp_iti_size.copy()
        spk_mat_resp = temp_resp_size.copy()
        # these are still trials x ms because that's what the grid is, but the actual values will be Hz
        sm_mat_stim  = temp_stim_size.copy()  # will hold smoothed rate: [trials x ms]
        sm_mat_iti   = temp_iti_size.copy()
        sm_mat_resp  = temp_resp_size.copy()

        # Now for any given unit, need to loop over all trials and cut windows around events,
        # for each trial, ii will grab fixation_time, stimulus_time etc (columns from trials_df), 
        # we see at the end of this loop, these times are used to slice from the spike train...
        for ii in range(n_trials):
            stim_st = round(trials_df['stimulus_time'].iloc[ii] * 1000) # stim onset in ms
            iti_st = round(trials_df['fixaton_time'].iloc[ii] * 1000) # fixation (in iti) onset in ms
            resp_st = round(trials_df['response_time'].iloc[ii] * 1000) # button press in ms

            # ..iti_end and response_time, however, are not involved in cutting the spike train,
            # they are just metadata we store in arrays, to later draw reference lines on the PSTH
            # as such, we only compute these once (jj = 0), because their timepoints don't change
            # so this is saying "only for first loop, fill iti_end (the blank array) with integers ii,
            # which are computed as (stimulus_time - fixation_time) * 1000 + pre_iti (buffer)"
            if jj == 0:
                iti_end[ii] = (trials_df['stimulus_time'].iloc[ii] - trials_df['fixaton_time'].iloc[ii]) * 1000 + pre_iti # stim - fixation is how long the ITI lasted
                response_time[ii] = (trials_df['response_time'].iloc[ii] - trials_df['stimulus_time'].iloc[ii]) * 1000 + pre_stim # response - stim is how long the response took

            # This is the actual slicing windows of spike_train and smooth_rate around each event
            # Each window is from event - pre, to event + post
            # e.g. from spk_mat_iti, row [ii] and all columns (i.e. :),
            # take spike_train and slice 250(or whatever) ms before iti_st and 750(or whatever) ms after
            # as stated prior, once these are filled, they're appended to the output dicts above

            spk_mat_stim[ii, :] = spike_train[stim_st - pre_stim : stim_st + post_stim]
            spk_mat_iti[ii, :] = spike_train[iti_st - pre_iti : iti_st + post_iti]
            spk_mat_resp[ii, :] = spike_train[resp_st - pre_resp : resp_st + post_resp]
            # and do the same for the smooth_rate... 
            sm_mat_stim[ii, :] = smooth_rate[stim_st - pre_stim : stim_st + post_stim]
            sm_mat_iti[ii,  :] = smooth_rate[iti_st  - pre_iti  : iti_st  + post_iti]
            sm_mat_resp[ii, :] = smooth_rate[resp_st - pre_resp : resp_st + post_resp]

        # Store this recording unit's results in the output dicts from earlier
        spk_stim_on['spk'].append(spk_mat_stim)
        spk_iti['spk'].append(spk_mat_iti)
        spk_resp['spk'].append(spk_mat_resp)

        spk_iti['spk_rate_sm'].append(sm_mat_iti)
        spk_stim_on['spk_rate_sm'].append(sm_mat_stim)
        spk_resp['spk_rate_sm'].append(sm_mat_resp)

        # Shuffling: we ask what a random, average firing rate looks like
        # We will use 97.5th and 2.5th percentiles as sig. thresholds.
        # first, we set the boundaries and create a blank array to store the mean firing rate from each shuffle
        if shuffle_fr:
            time_range_st = first_spike_time * 1000 + pre_stim # earliest valid window start (ms)
            time_range_end = last_spike_time * 1000 - post_stim # latest valid window start (ms)
            shuffle_means = np.zeros(x_shuffle) # blank 1d array with one slot per x_shuffle

            # for each shuffle kk...
            # random_starts: we draw n_trials random timestamps from within the valid time range,
            # np.random.uniform is (low, high, n_trials)
            # shuffle_windows: new blank 2d matrix (n_trials x ms), to be filled by the spike data
            for kk in range(x_shuffle):
                    random_starts = np.random.uniform(time_range_st, time_range_end, n_trials) 
                    shuffle_windows = np.zeros((n_trials, pre_stim + post_stim))
                    # loop through each of random_starts and cut windows from smooth_rate,
                    # the window being the same size as real windows from earlier
                    # each cut goes into one row of shuffle_windows
                    # need ii and r to delineate index vs actual timestamp
                    # here "r" is the random time value, st = round(r) converts the times to an integer
                    for ii, r in enumerate(random_starts):
                        st = round(r)
                        shuffle_windows[ii,:] = smooth_rate[st - pre_stim : st + post_stim]
                    
                    # takes the mean across all trials AND all timepoints into single number
                    # we output however many of these is in x_shuffle, and this becomes an array of chance firing rates
                    shuffle_means[kk] = np.mean(shuffle_windows)
            # we sort them so that we can find the sig. thresholds 
            sorted_means = np.sort(shuffle_means)
            # recall - shuffle_hist: if shuffle_fr = true this is the significance thresholds
            # dist: list of sorted means, pos: pos threshold (98th/100th value here),
            # and neg: neg threshold (2nd/100th value here)
            shuffle_hist[jj] = {
                'dist': sorted_means,
                'pos': sorted_means[alpha_p_threshold - 1],
                'neg': sorted_means[alpha_n_threshold - 1]
            }
    # lastly, we need to compute the trial-averaged mean and std of each epoch across all units
    # also need to mark where firing rate crosses significance for any given unit
    # this will use the find_positive_section function from below
    spk_stim_on = find_positive_section(spk_stim_on, shuffle_hist, sig_window)
    spk_iti = find_positive_section(spk_iti, shuffle_hist, sig_window)
    spk_resp = find_positive_section(spk_resp, shuffle_hist, sig_window)

    ######## TO-DO:
    # Translate findPositiveSection from matlab
    # Translate condition_parsing
    # Finish BASICPROC 

    return spk_iti, spk_stim_on, spk_resp, shuffle_hist, iti_end, response_time
        



        



In [24]:
# Now we define the find_positive_section fn from above
# sm_mean: average  smoothed firing rate across all trials for each unit
# Then compare average to shuffle sig. thresholds
# Remove anything shorter than sig_window

def find_positive_section(spk_dict, shuffle_hist, sig_window):
    from scipy import ndimage

    # reach into dict (spk_stim_on, spk_iti, or spk_resp) and find length of smoothed spk rate
    n_units = len(spk_dict['spk_rate_sm'])

    # trim 250ms from each edge, since Gauss smoothing messes with edges
    # then average across trials
    # axis = 0 averages rows (trials), ie averaging vertically, column by column
    # trimmed: using a list comprehension which is basically a compact loop
    # it says for jj, run through all rows (trials) and trim 250ms from each end
    trimmed = [spk_dict['spk_rate_sm'][jj][:, 250 : -250] for jj in range(n_units)] # -250 means "250 pos from the end"
    sm_mean = np.array([np.mean(t, axis = 0) for t in trimmed]) # these arrays shaped units x time
    sm_std = np.array([np.std(t, axis = 0) for t in trimmed])

    pos_section = np.zeros_like(sm_mean, dtype = bool) # the _like means use whatever size sm_mean already is
    neg_section = np.zeros_like(sm_mean, dtype = bool) # Recall bool fills it with T/F

    for jj in range(n_units):
        if shuffle_hist[jj] is None:
            continue # saying to continue/skip in case there are units with no shuffle

        # compare across whole time axis for each unit array..
        pos_section[jj] = sm_mean[jj] > shuffle_hist[jj]['pos']
        neg_section[jj] = sm_mean[jj] < shuffle_hist[jj]['neg']

        # need to find True values, equivalent to matlab's bwconncomp
        # then also need to eliminate blips less than the 30ms sig_window
        for section in [pos_section, neg_section]: #just means run this twice, once with pos once with neg
            # ndimage.label will take the series of T/F from pos or neg_section, and label as clusters
            # e.g. F, F, T, T, F, F, F, T, T.. turns into labels 0, 0, 1, 1, 0, 0, 0, 2, 2, 
            # and each set of 1s is a cluster. So there, n_clusters = 2
            labeled, n_clusters = ndimage.label(section[jj]) #ndimage.label is designed to return the labeled array and the n_clusters
            # the mask just ends up being a map of where each cluster is located.
            # so in example above, cluster_mask for cluster_id 1 is just F,F,T,T,F,F,F,F,F,F,etc..
            # we need to do this just to get the size of each cluster in ms
            for cluster_id in range(1, n_clusters + 1):
                cluster_mask = labeled == cluster_id
            # then, we cut any cluster less than the size of the sig_window!
                if cluster_mask.sum() < sig_window:
                    section[jj][cluster_mask] = False
    # add results to the spk_dict
    spk_dict['sm_mean'] = sm_mean
    spk_dict['sm_std'] = sm_std
    spk_dict['pos_section'] = pos_section
    spk_dict['neg_section'] = neg_section
    return spk_dict

    # to clarify, spk_dict here is a parameter of the function find_positive_section, but the spk_dict
    # is really just a placeholder, for whatever dictionary you are referencing in parse_trials,
    # e.g. in   spk_stim_on = find_positive_section(spk_stim_on, shuffle_hist, sig_window),
    # the spk_stim_on in this case is the dict that spk_dict will feed back into.


In [ ]:
# lastly we need to translate the conditionParsing file from matlab
# this will be fed whatever spk_dict from parse_trials, e.g., spk_iti, spk_stim_on, or spk_resp
# Condition == 1 is congruent, Condition == 2 is incongruent

def condition_parsing(spk_dict, trials_df, end_trials):
    # find columns and responsekey
    condition = trials_df['Condition'].values[:end_trials] # 1 or 2 as above
    response_key = trials_df['ResponseKey'].values[:end_trials] # ie 1, 2, 3

    n_units = len(spk_dict['spk_rate_sm'])

    # empty indices 
    cong_spk = [] # raw spikes, congruent trials
    incong_spk = [] # raw spikes, incongruent trials
    cong_spk_sm = [] # smoothed rate
    incong_spk_sm = [] # smoothed

    # .shape[1] means the 2nd dimension, ie gives nu ber columns, 
    # so this is putting however many columns (ms), into time_len
    time_len = spk_dict['sm_mean'].shape[1]

    # make blank matrices of units x ms, one for mean firing rate, one for std error 
    # need 4 matrices for each, the first will be all congruent or incongruent trials, 
    # and then the next 3 slots hold response keys 1, 2, 3 (whichever the pt pressed)
    cong_mean = [np.zeros((n_units, time_len)) for _ in range(4)]
    incong_mean = [np.zeros((n_units, time_len)) for _ in range(4)]
    cong_se = [np.zeros((n_units, time_len)) for _ in range(4)]
    incong_se = [np.zeros((n_units, time_len)) for _ in range(4)]

   
    for jj in range(n_units):

        # trim 250ms edge buffers
        # within spk_dict, take spk (list of matrices), for jj-th unit, all trials, trim off each edge
        raw = spk_dict['spk'][jj][:end_trials, 250:-250] # raw trials x time
        sm = spk_dict['spk_rate_sm'][jj][:end_trials, 250:-250] # smoothed

        # make boolean mask that is true for congruent trials, false if incong, based on condition above
        cong_mask = condition == 1
        incong_mask = condition == 2

        # Append to above indices
        # e.g. for each unit, filter "raw" to condition 1, then append. Need 3 extra blanks for response keys
        # these will be filled with units x ms matrices, broken down by button press 1, 2, or 3
        cong_spk.append([raw[cong_mask], None, None, None])
        incong_spk.append([raw[incong_mask], None, None, None])
        cong_spk_sm.append([sm[cong_mask], None, None, None])
        incong_spk_sm.append([sm[incong_mask], None, None, None])

        # Now we add the breakdown by which response key was pressed
        # enumerate over resp (1 2 or 3, the actual key pressed), kk is just the position index (0, 1, 2),
        # so we use kk to direct the loop to the "None" slots above
        # for reference, enumerate returns (index, value)
        for kk, resp in enumerate([1,2,3]):
            c_mask = cong_mask & (response_key == resp) # cong trials AND what button pressed
            i_mask = incong_mask & (response_key == resp) # incong trials AND what button
            # Now need to put the masks into the above indices
            # kk needs to be +1 because kk = 0 is the first slot in each index above,
            # and that first slot contains the full cong/incong mask list, so in this loop 
            # we want to fill each slot 2, 3, 4 of the indices with trials where
            # buttons 1, 2, or 3 were pressed, respectively
            cong_spk[jj][kk + 1] = raw[c_mask]
            incong_spk[jj][kk + 1] = raw[i_mask]
            cong_spk_sm[jj][kk + 1] = sm[c_mask]
            incong_spk_sm[jj][kk + 1] = sm[i_mask]
        
        # next, compute trial-averaged mean firing rate for any given unit,
        # for each of the 4 slots in the indices above
        for slot in range(4):
            c_data = cong_spk_sm[jj][slot] # take congruent data in slot "slot"
            i_data = incong_spk_sm[jj][slot] # same for incong data
            if len(c_data) > 0:
                # average down rows (axis 0) to give one value per timepoint
                # recall cong_mean from above is a list of 4 matrices (each units x ms)
                # so "slot" picks the matrix from cong_mean, jj is the unit, : is all columns (all ms)
                # slot goes first here because you have to dictate the matrix slot 1-4,
                # then add these means to that matrix.
                # whereas above, e.g. for cong_spk, jj is first because it's immediately within the jj loop.
                cong_mean[slot][jj, :] = np.mean(c_data, axis = 0)
                # same below, std error is std / sqrt
                # recall len() just applies to the first dimension, here it's trials (rows),
                # because way back at the beginning, temp_stim_size we'd defined as trials x ms.
                cong_se[slot][jj, :] = np.std(c_data, axis = 0) / np.sqrt(len(c_data))

            if len(i_data) > 0:
                incong_mean[slot][jj, :] = np.mean(i_data, axis=0)
                incong_se[slot][jj, :] = np.std(i_data,  axis=0) / np.sqrt(len(i_data))

    spk_dict['cong_spk'] = cong_spk
    spk_dict['incong_spk'] = incong_spk
    spk_dict['cong_sm'] = cong_spk_sm
    spk_dict['incong_sm'] = incong_spk_sm
    # recall the mean and Std Error are calculated based on the smoothed data
    spk_dict['cong_mean'] = cong_mean
    spk_dict['incong_mean'] = incong_mean
    spk_dict['cong_se'] = cong_se
    spk_dict['incong_se'] = incong_se
    return spk_dict

##!!!


